In [1]:
import kagglehub
import shutil
import pandas as pd

In [2]:
# REPLACE WITH YOUR LOCAL PATH
data_path = "/Users/trentbellinger/Desktop/CS 224W/Project/Data"

In [3]:
src = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
shutil.copytree(src, data_path, dirs_exist_ok = True)

'/Users/trentbellinger/Desktop/CS 224W/Project/Data'

In [4]:
acct = pd.read_csv(f"{data_path}/HI-Small_accounts.csv")
acct.rename(columns = {"Bank Name":"bank_name", "Bank ID":"bank_id", "Account Number":"account_number", 
                       "Entity ID":"entity_id", "Entity Name":"entity_name"}, 
            inplace = True)
acct.head()

,bank_name,bank_id,account_number,entity_id,entity_name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [5]:
txn = pd.read_csv(f"{data_path}/HI-Small_Trans.csv")
txn.rename(columns = {"Timestamp":"timestamp", "From Bank":"from_bank", "Account":"from_account", "To Bank":"to_bank", 
                     "Account.1":"to_account", "Amount Received":"amount_received", "Receiving Currency":"receiving_currency", 
                     "Amount Paid":"amount_paid", "Payment Currency":"payment_currency", "Payment Format":"payment_format", 
                     "Is Laundering":"is_laundering"},
                    inplace = True)
txn["timestamp"] = pd.to_datetime(txn["timestamp"])
txn = txn.sort_values("timestamp").reset_index(drop=True)
txn.head()

,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering
0,2022-09-01,121,8123FB9B0,121,8123FB9B0,47.64,Saudi Riyal,47.64,Saudi Riyal,Reinvestment,0
1,2022-09-01,25170,8095AF7C0,25170,8095AF7C0,3917.42,Canadian Dollar,3917.42,Canadian Dollar,Reinvestment,0
2,2022-09-01,25665,809A7D4B0,24779,809189BA0,97.49,Canadian Dollar,97.49,Canadian Dollar,Credit Card,0
3,2022-09-01,32317,800D4E490,12004,800D4E750,13939.05,Euro,13939.05,Euro,Wire,0
4,2022-09-01,1024,800C8D9D0,1024,800C8D9D0,10.37,Euro,10.37,Euro,Reinvestment,0


In [6]:
# Label each transaction with the laundering attempt it belongs to (from the Patterns file)
pattern_rows, laundering_id, laundering_type = [], -1, None
with open(f"{data_path}/HI-Small_Patterns.txt") as f:
    for line in f:
        line = line.strip()
        if line.startswith("BEGIN LAUNDERING ATTEMPT"):
            laundering_id += 1
            # e.g. "BEGIN LAUNDERING ATTEMPT - FAN-OUT:  Max 16-degree Fan-Out" -> "Fan-Out"
            laundering_type = line.split(" - ", 1)[1].split(":")[0].strip().title()
        elif line and not line.startswith("END LAUNDERING ATTEMPT"):
            pattern_rows.append(line.split(",") + [laundering_type, laundering_id])

txn_key = ["timestamp", "from_bank", "from_account", "to_bank", "to_account", "amount_received",
           "receiving_currency", "amount_paid", "payment_currency", "payment_format"]
patterns = pd.DataFrame(pattern_rows, columns=txn_key + ["is_laundering", "laundering_type", "laundering_id"])
patterns["timestamp"] = pd.to_datetime(patterns["timestamp"])
patterns[["from_bank", "to_bank"]] = patterns[["from_bank", "to_bank"]].astype(int)
patterns[["amount_received", "amount_paid"]] = patterns[["amount_received", "amount_paid"]].astype(float)
patterns["laundering_id"] = patterns["laundering_id"].astype("Int64")
assert not patterns.duplicated(txn_key).any(), "a transaction appears in more than one laundering attempt"

n_txn = len(txn)
txn = txn.merge(patterns[txn_key + ["laundering_type", "laundering_id"]], on=txn_key, how="left")
assert len(txn) == n_txn
print(f"{txn['laundering_id'].notna().sum()} of {len(patterns)} pattern transactions matched, "
      f"{txn['laundering_id'].nunique()} laundering attempts")
txn[txn["laundering_id"].notna()].head()

3209 of 3209 pattern transactions matched, 370 laundering attempts


,timestamp,from_bank,from_account,to_bank,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,laundering_type,laundering_id
43470,2022-09-01 00:03:00,1467,8013C4030,20,80BC62F10,58702.10,Yuan,58702.10,Yuan,ACH,1,Cycle,1
45786,2022-09-01 00:04:00,119,811C597B0,48309,811C599A0,34254.65,Saudi Riyal,34254.65,Saudi Riyal,ACH,1,Gather-Scatter,2
66096,2022-09-01 00:06:00,21174,800737690,12,80011F990,2848.96,Euro,2848.96,Euro,ACH,1,Fan-Out,0
334966,2022-09-01 00:43:00,12963,80296CA20,241869,80F390FC0,12562.68,Swiss Franc,12562.68,Swiss Franc,ACH,1,Gather-Scatter,4
363468,2022-09-01 01:34:00,116,80F1AF380,127593,80A817660,16502.02,Australian Dollar,16502.02,Australian Dollar,ACH,1,Fan-Out,7


In [7]:
df = pd.merge(
    txn, acct, 
    left_on = ["from_bank", "from_account"], 
    right_on = ["bank_id", "account_number"], 
    how = "left"
).drop(
    columns = ["bank_id", "account_number"]
).rename(
    columns = {"bank_name":"from_bank_name", "entity_id":"from_entity", "entity_name":"from_entity_name"}
)

df = pd.merge(
    df, acct, 
    left_on = ["to_bank", "to_account"], 
    right_on = ["bank_id", "account_number"], 
    how = "left"
).drop(
    columns = ["bank_id", "account_number"]
).rename(
    columns = {"bank_name":"to_bank_name", "entity_id":"to_entity", "entity_name":"to_entity_name"}
)[
    ["timestamp", "from_bank", "from_bank_name", "from_entity", "from_entity_name", "from_account", 
   "to_bank", "to_bank_name", "to_entity", "to_entity_name", "to_account", "amount_received", 
   "receiving_currency", "amount_paid", "payment_currency", "payment_format", "is_laundering", 
   "laundering_type", "laundering_id"]
].reset_index(drop = True)

df.head()

,timestamp,from_bank,from_bank_name,from_entity,from_entity_name,from_account,to_bank,to_bank_name,to_entity,to_entity_name,to_account,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,laundering_type,laundering_id
0,2022-09-01,121,Saudi Arabia Bank #6,800F1BC50,Partnership #38226,8123FB9B0,121,Saudi Arabia Bank #6,800F1BC50,Partnership #38226,8123FB9B0,47.64,Saudi Riyal,47.64,Saudi Riyal,Reinvestment,0,NaN,<NA>
1,2022-09-01,25170,Canada Bank #28,800C661B0,Sole Proprietorship #34849,8095AF7C0,25170,Canada Bank #28,800C661B0,Sole Proprietorship #34849,8095AF7C0,3917.42,Canadian Dollar,3917.42,Canadian Dollar,Reinvestment,0,NaN,<NA>
2,2022-09-01,25665,Canada Bank #25,800C5EFF0,Partnership #35725,809A7D4B0,24779,Canada Bank #1,800C723D0,Partnership #7715,809189BA0,97.49,Canadian Dollar,97.49,Canadian Dollar,Credit Card,0,NaN,<NA>
3,2022-09-01,32317,France Bank #348,8009281F0,Corporation #20251,800D4E490,12004,Italy Bank #53,80095E610,Corporation #20081,800D4E750,13939.05,Euro,13939.05,Euro,Wire,0,NaN,<NA>
4,2022-09-01,1024,France Bank #51,80097BF70,Corporation #20597,800C8D9D0,1024,France Bank #51,80097BF70,Corporation #20597,800C8D9D0,10.37,Euro,10.37,Euro,Reinvestment,0,NaN,<NA>


In [8]:
df.to_csv(f"{data_path}/full_data.csv", index = False)